### Lab 4.1: Cross-Entropy Loss

In this lab you will modify our linear classifier implementation to use the binary cross-entropy loss function.

In [1]:
import numpy as np
import torch

from palmerpenguins import load_penguins
from mlxtend.plotting import plot_decision_regions
from matplotlib import pyplot as plt

In [ ]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# tricky code to randomly shuffle the rows
df = df.sample(frac=1).reset_index(drop=True)

# select only two specices
df = df[(df['species']=='Adelie')|(df['species']=='Chinstrap')]

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert species labels to 0 and 1
y = df['species'].map({'Adelie':0,'Chinstrap':1}).values

X -= np.mean(X,axis=0)

X = torch.tensor(X).float()
y = torch.tensor(y).float()

### Exercises

1. Here is the binary cross entropy loss function:

$$L(z) = \log(1+\exp(-yz))$$

First you need to derive the derivative of this loss function.  Then, recalling that $z = \vec{w}^T\vec{x}+b$, derive the necessary derivatives $\partial L/\partial \vec{w}$ and $\partial L/\partial b$.

Implement these in the `train_step` function below and check that your model trains correctly on the dataset.

In [16]:
class Perceptron:
    def __init__(self,lr=1e-3):
        # store the learning rate
        self.lr = lr

        # initialize the weights to small, normally-distributed values
        self.w = torch.normal(mean=0, std=0.01, size=(2,))

        # initialize the bias to zero
        self.b = torch.zeros(1)

    def train_step(self,X:torch.Tensor,y:torch.Tensor) -> None:
        """ Update the weights using the cross-entropy loss.
            Arguments:
             x: data matrix of shape (N,2)
             y: labels of shape (N,) 
        """
        y = y*2 - 1
        z = X @ self.w + self.b
        dLdw = -y * torch.sigmoid(-y*z) * X.T
        dLdb = -y * torch.sigmoid(-y*z)
        self.w = self.w - self.lr * torch.mean(dLdw, dim=1)
        self.b = self.b - self.lr * torch.mean(dLdb, dim=0)
    
    def predict(self,X:torch.Tensor) -> torch.Tensor:
        """ Calculate model prediction for all data points.
            Arguments:
             X: data matrix of shape (N,2)   
            Returns:
             Predicted labels (-1 or 1) of shape (N,)
        """
        z = X @ self.w + self.b
        return torch.where(z>0,1,0)
    
    def score(self,X:torch.Tensor,y:torch.Tensor) -> torch.Tensor:
        """ Calculate model accuracy.
            Arguments:
             X: data matrix of shape (N,2)   
             y: labels of shape (N,)
            Returns:
             Accuracy score
        """
        pred = self.predict(X)
        return torch.mean((pred==y).float())


Run the following code to train the model and print out the accuracy at each step.

In [17]:
lr = 1e-1
epochs = 100
model = Perceptron(lr)
for i in range(epochs):
    model.train_step(X,y)
    print(f'step {i}: {model.score(X,y)}')

step 0: 0.836448609828949
step 1: 0.8691588640213013
step 2: 0.8785046935081482
step 3: 0.9112149477005005
step 4: 0.9158878326416016
step 5: 0.9299065470695496
step 6: 0.9299065470695496
step 7: 0.9299065470695496
step 8: 0.9299065470695496
step 9: 0.9299065470695496
step 10: 0.9299065470695496
step 11: 0.9299065470695496
step 12: 0.9345794320106506
step 13: 0.9345794320106506
step 14: 0.9345794320106506
step 15: 0.9345794320106506
step 16: 0.9345794320106506
step 17: 0.9345794320106506
step 18: 0.9345794320106506
step 19: 0.9392523169517517
step 20: 0.9392523169517517
step 21: 0.9392523169517517
step 22: 0.9392523169517517
step 23: 0.9392523169517517
step 24: 0.9392523169517517
step 25: 0.9439252614974976
step 26: 0.9439252614974976
step 27: 0.9439252614974976
step 28: 0.9439252614974976
step 29: 0.9439252614974976
step 30: 0.9439252614974976
step 31: 0.9439252614974976
step 32: 0.9439252614974976
step 33: 0.9439252614974976
step 34: 0.9439252614974976
step 35: 0.9439252614974976
ste